# BASELINE · HierCon-Mimic — Hierarchical Contrastive Attention (arXiv:2602.01032)

Reimplementation-from-paper-description of:
> Liang, Han, Wang, Leckie. **"HierCon: Hierarchical Contrastive Attention for Audio
> Deepfake Detection."** arXiv:2602.01032. Official code: https://github.com/adlnlp/HierCon

**What the paper does (and what we mimic):**
- Backbone: **XLS-R 300M**, hidden states from **all 24 transformer layers**, **4 s** windows
- Stage-1: temporal attention pooling per layer (frames → layer embedding)
- Stage-2: neighbouring-layer attention (local inter-layer dependencies)
- Stage-3: layer-group attention (groups of layers → utterance embedding)
- Loss: BCE + **margin-based contrastive** on utterance embeddings (domain-invariance)
- Protocol: train ASVspoof **2019 LA**, eval **2021 LA / 2021 DF / In-the-Wild**, metric **EER**
- Reported: **1.93% EER (21-DF)**, **6.87% EER (ITW)**

**⚠️ Fidelity disclaimer (put this in the paper too):** this is a faithful-in-spirit
reimplementation from the paper text. Exact attention parameterization, margins, LR
schedule, and whether/which backbone layers are fine-tuned should be verified against
the official repo before you claim "reproduced". In the paper, cite THEIR reported
numbers for the comparison row and mark your reimplementation separately.

**Kaggle note:** full-backbone fine-tuning of XLS-R 300M on 19-LA is heavy for one P100
session. Default here: `FREEZE_BACKBONE=True` (train attention+head only, ~fits one
session). Flip to False + unfreeze top-N layers if you have budget.

In [ ]:
import os, glob, json, math, random, time, warnings
from pathlib import Path
import numpy as np
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "soundfile", "librosa"], check=False)
import torch, torch.nn as nn, torch.nn.functional as F
import soundfile as sf, librosa
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

class CFG:
    DEBUG            = True
    SEED             = 0
    SR               = 16000
    CROP_SEC         = 4.0            # paper: 4-second windows
    BACKBONE         = "facebook/wav2vec2-xls-r-300m"   # paper backbone
    N_LAYERS         = 24             # transformer layers used
    LAYER_GROUPS     = 4              # 24 layers -> 4 groups of 6 (design choice; verify vs repo)
    D_ATT            = 256
    FREEZE_BACKBONE  = True           # paper fine-tunes; True = Kaggle-budget mode
    UNFREEZE_TOP_N   = 0              # if FREEZE_BACKBONE=False, unfreeze only top-N layers
    BATCH            = 8
    EPOCHS           = 4
    LR_HEAD          = 1e-4
    LR_BACKBONE      = 1e-5
    MARGIN           = 1.0            # contrastive margin (verify vs repo)
    LAMBDA_CON       = 0.5            # BCE + λ·contrastive (verify vs repo)
    # ---- ASVspoof 2019 LA layout (attach a Kaggle mirror and fix these) ----
    LA_ROOT          = "/kaggle/input/asvspoof-2019-dataset/LA"   # common Kaggle mirror layout
    TRAIN_AUDIO      = "ASVspoof2019_LA_train/flac"
    DEV_AUDIO        = "ASVspoof2019_LA_dev/flac"
    EVAL_AUDIO       = "ASVspoof2019_LA_eval/flac"
    PROTO_DIR        = "ASVspoof2019_LA_cm_protocols"
    OUT              = Path("/kaggle/working")

if CFG.DEBUG:
    CFG.EPOCHS = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG.SEED)
print(DEVICE)

## Data — ASVspoof 2019 LA protocol parsing (bonafide=0, spoof=1)

In [ ]:
def parse_protocol(name, audio_subdir):
    proto = Path(CFG.LA_ROOT) / CFG.PROTO_DIR / name
    audio = Path(CFG.LA_ROOT) / audio_subdir
    items = []
    for line in proto.read_text().strip().splitlines():
        parts = line.split()
        utt, key = parts[1], parts[-1]           # SPK UTT ... bonafide|spoof
        f = audio / f"{utt}.flac"
        if f.exists():
            items.append((str(f), 0 if key == "bonafide" else 1))
    print(f"{name}: {len(items)} files  (spoof={sum(l for _,l in items)})")
    return items

TRAIN = parse_protocol("ASVspoof2019.LA.cm.train.trn.txt", CFG.TRAIN_AUDIO)
DEV   = parse_protocol("ASVspoof2019.LA.cm.dev.trl.txt",   CFG.DEV_AUDIO)
EVAL  = parse_protocol("ASVspoof2019.LA.cm.eval.trl.txt",  CFG.EVAL_AUDIO)
if CFG.DEBUG:
    random.shuffle(TRAIN); random.shuffle(DEV); random.shuffle(EVAL)
    TRAIN, DEV, EVAL = TRAIN[:400], DEV[:200], EVAL[:200]

def load_clip(path, sr=CFG.SR, crop=CFG.CROP_SEC, train=False):
    y, fsr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim > 1: y = y.mean(1)
    if fsr != sr: y = librosa.resample(y, orig_sr=fsr, target_sr=sr)
    T = int(sr * crop)
    if len(y) >= T:
        off = random.randint(0, len(y) - T) if train else (len(y) - T) // 2
        y = y[off:off+T]
    else:
        reps = math.ceil(T / max(len(y), 1)); y = np.tile(y, reps)[:T]   # repeat-pad (common in ADD)
    m = np.abs(y).max(); return y / m if m > 0 else y

class ADDataset(torch.utils.data.Dataset):
    def __init__(self, items, train=False): self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, l = self.items[i]
        return torch.from_numpy(load_clip(p, train=self.train)), l

## Model — hierarchical attention over (frames → neighbouring layers → layer groups)

In [ ]:
from transformers import Wav2Vec2Model

class AttnPool(nn.Module):
    """Additive attention pooling over an axis."""
    def __init__(self, d, d_att=CFG.D_ATT):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d_att), nn.Tanh(), nn.Linear(d_att, 1))
    def forward(self, x):                       # x: [B, N, D]
        a = torch.softmax(self.w(x), dim=1)     # [B, N, 1]
        return (a * x).sum(1), a.squeeze(-1)    # [B, D], [B, N]

class HierConMimic(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained(CFG.BACKBONE)
        if CFG.FREEZE_BACKBONE:
            for p in self.backbone.parameters(): p.requires_grad = False
        elif CFG.UNFREEZE_TOP_N > 0:
            for p in self.backbone.parameters(): p.requires_grad = False
            for layer in self.backbone.encoder.layers[-CFG.UNFREEZE_TOP_N:]:
                for p in layer.parameters(): p.requires_grad = True
        D = self.backbone.config.hidden_size    # 1024 for XLS-R 300M
        # Stage 1: temporal attention per layer (shared weights across layers)
        self.temporal = AttnPool(D)
        # Stage 2: neighbouring-layer attention — local mixing across adjacent layers
        self.local_mix = nn.Conv1d(D, D, kernel_size=3, padding=1, groups=1)
        # Stage 3a: within-group attention; 3b: across-group attention
        self.group_pool  = AttnPool(D)
        self.global_pool = AttnPool(D)
        self.proj = nn.Sequential(nn.Linear(D, 256), nn.ReLU(), nn.Linear(256, 128))  # contrastive space
        self.head = nn.Linear(D, 1)

    def forward(self, wav):                     # wav: [B, T]
        ctx = torch.no_grad() if CFG.FREEZE_BACKBONE else torch.enable_grad()
        with ctx:
            out = self.backbone(wav, output_hidden_states=True)
        hs = torch.stack(out.hidden_states[1:1+CFG.N_LAYERS], dim=1)   # [B, L=24, T', D]
        B, L, T, D = hs.shape
        # Stage 1: frames -> per-layer embedding
        layer_emb, _ = self.temporal(hs.reshape(B*L, T, D))            # [B*L, D]
        layer_emb = layer_emb.view(B, L, D)                            # [B, L, D]
        # Stage 2: neighbouring-layer dependencies (residual local conv over the layer axis)
        layer_emb = layer_emb + self.local_mix(layer_emb.transpose(1, 2)).transpose(1, 2)
        # Stage 3: layer groups -> group embeddings -> utterance embedding
        G = CFG.LAYER_GROUPS; per = L // G
        groups = layer_emb[:, :G*per].view(B, G, per, D)
        g_emb, _ = self.group_pool(groups.reshape(B*G, per, D))
        g_emb = g_emb.view(B, G, D)
        utt, group_attn = self.global_pool(g_emb)                      # [B, D]
        return self.head(utt).squeeze(-1), F.normalize(self.proj(utt), dim=-1), group_attn

def margin_contrastive(z, y, margin=CFG.MARGIN):
    """Pairwise margin contrastive on normalized embeddings.
    Same class -> pull (d^2); different class -> push (max(0, m-d)^2). In-batch pairs."""
    d = torch.cdist(z, z)                                   # [B, B]
    same = (y[:, None] == y[None, :]).float()
    eye = torch.eye(len(y), device=z.device)
    pos = same * (1 - eye)
    neg = 1 - same
    l_pos = (pos * d.pow(2)).sum() / pos.sum().clamp(min=1)
    l_neg = (neg * F.relu(margin - d).pow(2)).sum() / neg.sum().clamp(min=1)
    return l_pos + l_neg

## Metrics — EER (the paper's metric)

In [ ]:
def compute_eer(y, scores):
    y, scores = np.asarray(y), np.asarray(scores)
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2)

@torch.no_grad()
def evaluate(model, items, name):
    model.eval()
    dl = torch.utils.data.DataLoader(ADDataset(items), batch_size=CFG.BATCH, num_workers=2)
    ss, yy = [], []
    for x, y in dl:
        logit, _, _ = model(x.to(DEVICE))
        ss.append(torch.sigmoid(logit).cpu().numpy()); yy.append(y.numpy())
    s, y = np.concatenate(ss), np.concatenate(yy)
    res = {"eer": compute_eer(y, s), "auc": float(roc_auc_score(y, s)), "n": int(len(y))}
    print(f"[eval:{name}] EER={res['eer']*100:.2f}%  AUC={res['auc']:.4f}  n={res['n']}")
    return res, s, y

## Train — BCE + λ·margin-contrastive on ASVspoof 2019 LA

In [ ]:
model = HierConMimic().to(DEVICE)
params = [{"params": [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone")],
           "lr": CFG.LR_HEAD}]
if not CFG.FREEZE_BACKBONE:
    params.append({"params": [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone")],
                   "lr": CFG.LR_BACKBONE})
opt = torch.optim.AdamW(params, weight_decay=1e-4)
dl = torch.utils.data.DataLoader(ADDataset(TRAIN, train=True), batch_size=CFG.BATCH,
                                 shuffle=True, num_workers=2, drop_last=True)
best_dev = 1.0
for ep in range(CFG.EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    for step, (x, y) in enumerate(dl):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logit, z, _ = model(x)
        loss = F.binary_cross_entropy_with_logits(logit, y.float()) + CFG.LAMBDA_CON * margin_contrastive(z, y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); tot += loss.item()
        if (step+1) % 100 == 0: print(f"  ep{ep+1} step{step+1}/{len(dl)} loss={tot/(step+1):.4f}")
    dev_res, _, _ = evaluate(model, DEV, "19LA-dev")
    print(f"[ep{ep+1}] loss={tot/len(dl):.4f} dev-EER={dev_res['eer']*100:.2f}%  ({time.time()-t0:.0f}s)")
    if dev_res["eer"] < best_dev:
        best_dev = dev_res["eer"]
        torch.save(model.state_dict(), CFG.OUT / "hiercon_mimic_best.pt")
        print("  ↳ saved best")

## Evaluate — 19-LA eval now; 21-DF / ITW when those datasets are attached

Reference targets from the paper: 21-DF **1.93% EER**, ITW **6.87% EER** (with full
fine-tuning). A frozen-backbone mimic WILL be worse — report it as
"HierCon-style (frozen, reimpl.)" and cite their numbers as the official row.

In [ ]:
model.load_state_dict(torch.load(CFG.OUT / "hiercon_mimic_best.pt", map_location=DEVICE))
results = {"paper": {"citation": "Liang et al., HierCon, arXiv:2602.01032",
                     "reported_eer": {"asvspoof21_df": 0.0193, "in_the_wild": 0.0687}},
           "mimic": {"config": {k: v for k, v in vars(CFG).items()
                                if not k.startswith("_") and isinstance(v, (int, float, str, bool))}}}
r, s, y = evaluate(model, EVAL, "19LA-eval")
results["mimic"]["19la_eval"] = r
np.save(CFG.OUT / "hiercon_scores_19la_eval.npy", np.stack([s, y]))

# Optional cross-corpus hooks — set globs when datasets are attached:
EXTRA = {
    "in_the_wild": ("/kaggle/input/release-in-the-wild/**/real/**/*.wav",
                    "/kaggle/input/release-in-the-wild/**/fake/**/*.wav"),
    # "asvspoof21_df_subset": ("<real glob>", "<fake glob>"),
}
for name, (rg, fg) in EXTRA.items():
    reals = sorted(glob.glob(rg, recursive=True)); fakes = sorted(glob.glob(fg, recursive=True))
    if not (reals and fakes):
        print(f"[skip] {name}: dataset not attached"); continue
    if CFG.DEBUG: reals, fakes = reals[:150], fakes[:150]
    items = [(p, 0) for p in reals] + [(p, 1) for p in fakes]
    r, s, y = evaluate(model, items, name)
    results["mimic"][name] = r
    np.save(CFG.OUT / f"hiercon_scores_{name}.npy", np.stack([s, y]))

json.dump(results, open(CFG.OUT / "hiercon_mimic_results.json", "w"), indent=2)
print(json.dumps(results["mimic"], indent=2))